In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/malakwahba/resume-cv/Malak Wahba (4).pdf


In [2]:
!pip install -U langchain langchain-community langchain-core transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 114.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import torch
import re

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

def generate_text(prompt, max_new_tokens=500, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )

    return [
        tokenizer.decode(output, skip_special_tokens=True)
        for output in outputs
    ]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [6]:
full_name_schema = ResponseSchema(
    name="full_name",
    description="The full name of the candidate."
)

email_schema = ResponseSchema(
    name="email",
    description="The email address of the candidate."
)

education_schema = ResponseSchema(
    name="education",
    description="A list of education entries, where each entry contains degree, institution, and year."
)

skills_schema = ResponseSchema(
    name="skills",
    description="A list of skills possessed by the candidate."
)

experience_schema = ResponseSchema(
    name="experience",
    description="A list of work experience entries, where each entry contains role, company, and years."
)

response_schemas = [
    full_name_schema,
    email_schema,
    education_schema,
    skills_schema,
    experience_schema
]

output_parser = StructuredOutputParser.from_response_schemas(
    response_schemas
)

format_instructions = output_parser.get_format_instructions()

In [7]:
print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The full name of the candidate.
	"email": string  // The email address of the candidate.
	"education": string  // A list of education entries, where each entry contains degree, institution, and year.
	"skills": string  // A list of skills possessed by the candidate.
	"experience": string  // A list of work experience entries, where each entry contains role, company, and years.
}
```


In [8]:
cv_extraction_template = """

You are a smart assistant that extracts structured information from a candidate's resume.

Extract:
- full name
- email
- education
- skills
- experience

For education, extract degree, institution, and year.

For experience, extract role, company, and years.

Respond ONLY in JSON format:

{format_instructions}

Resume:

{cv_text}

"""

prompt = cv_extraction_template.format(
    format_instructions=format_instructions,
    cv_text=cv_text
)

In [9]:
prompt = PromptTemplate(
    template=cv_extraction_template,
    input_variables=["cv_text", "format_instructions"]
).format(
    cv_text=cv_text,
    format_instructions=format_instructions
)


In [10]:
response = generate_text(prompt, max_new_tokens=500)[0]

print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.




You are a smart assistant that extracts structured information from a candidate's resume.

Extract:
- full name
- email
- education
- skills
- experience

For education, extract degree, institution, and year.

For experience, extract role, company, and years.

Respond ONLY in JSON format:

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The full name of the candidate.
	"email": string  // The email address of the candidate.
	"education": string  // A list of education entries, where each entry contains degree, institution, and year.
	"skills": string  // A list of skills possessed by the candidate.
	"experience": string  // A list of work experience entries, where each entry contains role, company, and years.
}
```

Resume:

Malak Alaa El-Din Wahba         
Cairo, Egypt • +20 100 326 2695 • malakalaawahba@gmail.com • linkedin.com/in/malak-wahba     
Summary     

In [13]:
test = generate_text(prompt, max_new_tokens=500)

print(type(test))
print(test)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<class 'list'>
['\n\nYou are a smart assistant that extracts structured information from a candidate\'s resume.\n\nExtract:\n- full name\n- email\n- education\n- skills\n- experience\n\nFor education, extract degree, institution, and year.\n\nFor experience, extract role, company, and years.\n\nRespond ONLY in JSON format:\n\nThe output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{\n\t"full_name": string  // The full name of the candidate.\n\t"email": string  // The email address of the candidate.\n\t"education": string  // A list of education entries, where each entry contains degree, institution, and year.\n\t"skills": string  // A list of skills possessed by the candidate.\n\t"experience": string  // A list of work experience entries, where each entry contains role, company, and years.\n}\n```\n\nResume:\n\nMalak Alaa El-Din Wahba         \nCairo, Egypt • +20 100 326 2695 • malakalaawahba@gm

In [18]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"

In [21]:
def parse_cv(pdf_path):

    # 1. Load PDF
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # 2. Extract CV text
    cv_text = "\n".join(
        doc.page_content for doc in documents
    )

    # 3. Create prompt
    prompt = PromptTemplate(
        template=cv_extraction_template,
        input_variables=["cv_text", "format_instructions"]
    ).format(
        cv_text=cv_text,
        format_instructions=format_instructions
    )

    # 4. Generate response
    response = generate_text(
        prompt,
        max_new_tokens=500
    )[0]   # <-- IMPORTANT: convert list output to string

    print("Response type:", type(response))

    # 5. Extract JSON
    json_text = extract_json_block(response)

    # 6. Remove markdown code fences
    clean_json = json_text.replace(
        "```json", ""
    ).replace(
        "```", ""
    ).strip()

    # 7. Convert JSON string to Python dictionary
    import json
    result = json.loads(clean_json)

    return result

In [22]:
result = parse_cv(
    "/kaggle/input/datasets/malakwahba/resume-cv/Malak Wahba (4).pdf"
)

print(result)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Response type: <class 'str'>
{'full_name': 'Malak Alaa El-Din Wahba', 'email': 'malakalaawahba@gmail.com', 'education': [{'degree': 'B.Sc. in Business Informatics', 'institution': 'German University in Cairo', 'year': '2027'}, {'degree': 'American Diploma', 'institution': 'Qimam El Hayat International School at Riyadh', 'year': '2023'}], 'skills': ['Python', 'SQL', 'Machine Learning', 'Power BI', 'DAX', 'SAP ERP', 'Java', 'Data Analysis', 'React', 'Git', 'HTML', 'CSS'], 'experience': [{'role': 'AI Engineering Intern', 'company': 'Edrak', 'years': 'Sep 2026 – Present'}, {'role': 'Management Consulting Intern', 'company': 'Forvis Mazars', 'years': 'Aug 2026 – Sep 2026'}, {'role': 'Digital Banking Intern', 'company': 'Commercial International Bank (CIB)', 'years': 'Aug 2026 – Aug 2026'}, {'role': 'Data Scientist Intern', 'company': 'Arab African International Bank', 'years': 'Jul 2026 – Aug 2026'}, {'role': 'Information Technology Intern', 'company': 'Badr El-Din Petroleum Company', 'year

In [ ]:
json_text = extract_json_block(response)

In [ ]:
print(json_text)

In [24]:
!pip install fastapi uvicorn pyngrok transformers==4.52.4 accelerate -q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [25]:
import fastapi
import uvicorn
import pyngrok
import transformers
import accelerate

print("All packages installed successfully!")

All packages installed successfully!


In [26]:
from fastapi import FastAPI, UploadFile, File, HTTPException
import tempfile
import os

app = FastAPI()

@app.post("/parse-cv")
async def parse_cv_endpoint(file: UploadFile = File(...)):

    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(
            status_code=400,
            detail="Only PDF files are supported"
        )

    with tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".pdf"
    ) as temp:
        temp.write(await file.read())
        pdf_path = temp.name

    try:
        result = parse_cv(pdf_path)
        return result

    finally:
        if os.path.exists(pdf_path):
            os.remove(pdf_path)

In [28]:
from pyngrok import ngrok, conf

NGROK_TOKEN = "3KDa5VCpmXUDxitMxonHfLoHyQQ_2GoGLSNRExhfuZ3R35gCU"

conf.get_default().auth_token = NGROK_TOKEN

In [29]:
import uvicorn
import threading
import socket
import time
from pyngrok import ngrok

# Find an available port
def free_port():
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()

# Start FastAPI
def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=port
    )

threading.Thread(
    target=run,
    daemon=True
).start()

time.sleep(2)

# Create ngrok tunnel
public_url = ngrok.connect(port).public_url

print("Public URL:", public_url)
print("API endpoint:", public_url + "/parse-cv")

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:52893 (Press CTRL+C to quit)


Public URL: https://private-barstool-refueling.ngrok-free.dev                                       
API endpoint: https://private-barstool-refueling.ngrok-free.dev/parse-cv


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Response type: <class 'str'>
INFO:     102.43.141.26:0 - "POST /parse-cv HTTP/1.1" 200 OK
